# RFSoC-SAM I/Q capture experiments

Legacy experiments using the RFSoC-SAM spectrum analyser and the separate
RFSoC QPSK overlay. These cells do not use the ADS-B capture overlay in
`pynq/adsb_capture/`.


In [ ]:
from rfsoc_sam.overlay import Overlay
import numpy as np

sam = Overlay()

# Spectrum-analyser channel and DMA endpoint.
channel = sam.radio.receiver.channels[0]
sa = channel.spectrum_analyser

# Automatic DMA enabled for processed spectrum frames.
sa.dma_enable = 1

## Spectrum-analyser DMA frame


In [ ]:
frame = sa.get_frame()
print(frame.shape)
print(frame.min())
print(frame.max())
print(frame[:10])  # First ten bins

Observed RFSoC-SAM noise-floor values were approximately -90 to -105 dBFS.
The returned frame contains processed spectrum values rather than time-domain
I/Q samples.


---


## Spectrum-frame storage


In [ ]:
from pathlib import Path

output_path = Path("/home/xilinx/captures/spectrum_frame.npy")
output_path.parent.mkdir(parents=True, exist_ok=True)
np.save(output_path, frame)

The spectrum-analyser frame contains 2048 floating-point power-spectrum bins
in dBFS.

Recorded data path:

```text
RF-ADC -> DDC -> SSR conversion -> 2048-point FFT
       -> spectrum processor -> automatic DMA -> float32 frame
```

The processing before DMA is implemented in programmable logic. Changes to
that path, including its sample-rate configuration, require a different
bitstream.


---


# QPSK overlay I/Q path

The Strathclyde QPSK overlay exposes a DMA buffer on a decimated I/Q path.
This section records its suitability for raw-sample extraction.


In [ ]:
from rfsoc_qpsk.qpsk_overlay import QpskOverlay
import numpy as np

qpsk = QpskOverlay()

RFDC mixer configuration is exposed through the `xrfdc` driver.


In [ ]:
import xrfdc

mixer_cfg = qpsk.adc_block.MixerSettings
mixer_cfg['Freq'] = 1090 #MHz
qpsk.adc_block.MixerSettings = mixer_cfg
qpsk.adc_block.UpdateEvent(xrfdc.EVENT_MIXER)

print("ADC mixer frequency: 1090 MHz")

A single `get_decimated()` call returns 128 complex samples.


In [ ]:
frameIQ = qpsk.qpsk_rx.get_decimated()
print(frameIQ.shape)
print(frameIQ.dtype)
print(frameIQ[:5])  # First five samples

The recorded RF-ADC rate was 1024 MSPS with RFDC decimation by 8, producing
128 MSPS complex output.

ADS-B extended squitter uses 1 Mbit/s pulse-position modulation, a 0.5 us
pulse width, an 8 us preamble, and a total long-message duration of 120 us.


In [ ]:
print(qpsk.adc_block.BlockStatus)
print(qpsk.adc_block.DecimationFactor)

The 128-sample DMA frame spans 1 us at 128 MSPS. Approximately 120 frames are
therefore required to cover a 120 us ADS-B extended-squitter message.

The recorded experiment excluded the first 15 frames because of a repeatable
startup transient. The resulting capture is shorter than a complete message.


In [ ]:
# A repeatable transient occurs near 5 us in each acquisition.
# The first 15 frames are excluded pending identification of its source.

N_frames = 120
framesInt = [qpsk.qpsk_rx.get_decimated() for _ in range (N_frames)]
iq_data = np.concatenate(framesInt[15:])
print("Captured samples: ", iq_data.shape)
print("Duration: ", len(iq_data)/128e6 * 1e6, "us")

In [ ]:
import matplotlib.pyplot as plt
t = np.arange(len(iq_data)) / 128e6 * 1e6

plt.figure(figsize=(12,4))
plt.plot(t, np.real(iq_data), label='I')
plt.plot(t, np.imag(iq_data), label='Q')
plt.xlabel('Time (us)')
plt.ylabel('Amp')
plt.title('Raw IQ at 1090 MHz')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Preliminary spectrum calculation retained for comparison.

# iq_clean = iq_data - np.mean(iq_data)  # Optional DC removal
iq_clean = iq_data
N = len(iq_clean)
fftRaw = np.fft.fft(iq_clean) # Consider iq_data / N to normalise 
fft_norm = np.fft.fftshift(fftRaw) / N
freqs = np.fft.fftshift(np.fft.fftfreq(N, d=1/128e6)) / 1e6 # MHz offset from 1090MHz

power = 20*np.log10(np.abs(fft_norm) + 1e-10)
peak_bin = np.argmax(power)
print(peak_bin)
print(freqs[peak_bin])
print(power[peak_bin])
print(N//2)
print(fft_norm[N//2])
print("\nBins around peak: ")
for i in range(peak_bin-5, peak_bin+5):
    print(f" bin {i}: {power[i]:.2f} dB")

print(iq_data.shape)  # Diagnostic output
print(np.max(np.abs(iq_data)))  # Diagnostic output
print(iq_data.dtype)  # Diagnostic output
print("FFT MAX VALUE: ", 20*np.log10(np.max(np.abs(fft_norm))))


centre_bin = N//2
fft_norm[centre_bin] = (fft_norm[centre_bin - 1] + fft_norm[centre_bin + 1]) / 2

plt.figure(figsize=(12,4))
plt.plot(freqs, 20*np.log10(np.abs(fft_norm) + 1e-10))

plt.xlabel('Freq offset from 1090 MHz (MHz)')
plt.ylabel('Power (dB)')
plt.title('Spectrum @ 1090Mhz')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
qpsk.qpsk_tx.qpsk_tx.enable = 0